In [ ]:
# =============================================================================
# CELL 1: CONFIGURATION
# =============================================================================

# Match this to your notebook's START_DATE / min_date_sql
MIN_BOOK_DATE = "'2026-10-01'"   # must keep the inner SQL quotes

DSN           = "Redshift_prod_new"
STOP_ON_ERROR = False            # True = halt on first failure; False = run all steps

In [ ]:
# =============================================================================
# CELL 2: IMPORTS + HELPER
# =============================================================================

import warnings
import pyodbc
import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)

def run_step(step_id, description, sql, conn):
    """Run one diagnostic query and display the result."""
    query = sql.replace('{min_book_date}', MIN_BOOK_DATE)
    print(f"\n{'─' * 70}")
    print(f"  STEP {step_id}: {description}")
    print(f"{'─' * 70}")
    try:
        warnings.filterwarnings('ignore', category=UserWarning)
        df = pd.read_sql_query(query, conn)
        warnings.filterwarnings('default', category=UserWarning)
        display(df)
        print(f"  ✔  OK ({len(df)} row(s))")
        return True
    except Exception as exc:
        warnings.filterwarnings('default', category=UserWarning)
        print(f"  ✘  FAILED: {exc}")
        return False

# Open one shared connection for the whole notebook
conn = pyodbc.connect(f"DSN={DSN}")
print(f"Connected to {DSN}")
print(f"min_book_date = {MIN_BOOK_DATE}")

In [ ]:
# =============================================================================
# STEP 1: CTE SANITY CHECKS
# Validate each CTE compiles and returns rows in isolation.
# =============================================================================

run_step('1a', 'student_loan CTE — compiles and returns rows', """
    SELECT COUNT(*) AS student_loan_row_count
    FROM (
        SELECT
            loan_id,
            MAX(federal_student_loan_flag) AS student_loan_flag,
            MAX(chime_flag)                AS sandbox_chime
        FROM sandbox.student_loan_chime_flags
        GROUP BY loan_id
    ) t
""", conn)

run_step('1b', 'employment_agg CTE — compiles and returns rows', """
    SELECT COUNT(*) AS employment_agg_row_count
    FROM (
        SELECT
            account_number,
            MAX(CASE WHEN employment_type = 'seasonal' THEN 1 ELSE 0 END) AS seasonal_flag,
            MAX(CASE WHEN employment_type = 'waiter'   THEN 1 ELSE 0 END) AS waiter_flag
        FROM sandbox.temp_employment_type_ragu
        GROUP BY account_number
    ) t
""", conn)

In [ ]:
# =============================================================================
# STEP 2: BASE TABLE ROW COUNT
# Confirms the date filter resolves correctly and volume is reasonable.
# =============================================================================

run_step('2', 'Base table row count with date filter', """
    SELECT COUNT(*) AS base_row_count
    FROM edwnpi.los_deal_current_fact AS cd
    WHERE cd.data_source_name != 'SPARTAN'
      AND cd.book_date IS NOT NULL
      AND cd.book_date >= {min_book_date}
""", conn)

In [ ]:
# =============================================================================
# STEP 3: INCREMENTAL JOIN CARDINALITY
# Each sub-step adds one JOIN. A sudden count jump flags a fan-out on that key.
# Compare each result to the previous — they should stay roughly equal.
# =============================================================================

cardinality_steps = [
    ('3a', '+ student_loan (loan_id)', """
        SELECT COUNT(*) AS after_student_loan_join
        FROM edwnpi.los_deal_current_fact AS cd
        LEFT JOIN (
            SELECT loan_id,
                   MAX(federal_student_loan_flag) AS student_loan_flag,
                   MAX(chime_flag)                AS sandbox_chime
            FROM sandbox.student_loan_chime_flags
            GROUP BY loan_id
        ) stu ON stu.loan_id = cd.loan_id
        WHERE cd.data_source_name != 'SPARTAN'
          AND cd.book_date IS NOT NULL
          AND cd.book_date >= {min_book_date}
    """),
    ('3b', '+ dealer_rollup (dealer_number)', """
        SELECT COUNT(*) AS after_dealer_rollup_join
        FROM edwnpi.los_deal_current_fact AS cd
        LEFT JOIN (
            SELECT loan_id,
                   MAX(federal_student_loan_flag) AS student_loan_flag,
                   MAX(chime_flag)                AS sandbox_chime
            FROM sandbox.student_loan_chime_flags
            GROUP BY loan_id
        ) stu ON stu.loan_id = cd.loan_id
        LEFT JOIN edwnpi.dealer_rollup_scd_current AS dru
            ON dru.dealer_number = cd.dealer_number
        WHERE cd.data_source_name != 'SPARTAN'
          AND cd.book_date IS NOT NULL
          AND cd.book_date >= {min_book_date}
    """),
    ('3c', '+ blackbook rollup (account_number) [no uniqueness guarantee]', """
        SELECT COUNT(*) AS after_blackbook_join
        FROM edwnpi.los_deal_current_fact AS cd
        LEFT JOIN (
            SELECT loan_id,
                   MAX(federal_student_loan_flag) AS student_loan_flag,
                   MAX(chime_flag)                AS sandbox_chime
            FROM sandbox.student_loan_chime_flags
            GROUP BY loan_id
        ) stu ON stu.loan_id = cd.loan_id
        LEFT JOIN edwnpi.dealer_rollup_scd_current AS dru
            ON dru.dealer_number = cd.dealer_number
        LEFT JOIN sandbox.rds_blackbook_rollup bbvt
            ON bbvt.account_number = cd.account_number
        WHERE cd.data_source_name != 'SPARTAN'
          AND cd.book_date IS NOT NULL
          AND cd.book_date >= {min_book_date}
    """),
    ('3d', '+ kmx_approvals (loan_id)', """
        SELECT COUNT(*) AS after_kmx_approvals_join
        FROM edwnpi.los_deal_current_fact AS cd
        LEFT JOIN (
            SELECT loan_id,
                   MAX(federal_student_loan_flag) AS student_loan_flag,
                   MAX(chime_flag)                AS sandbox_chime
            FROM sandbox.student_loan_chime_flags
            GROUP BY loan_id
        ) stu ON stu.loan_id = cd.loan_id
        LEFT JOIN edwnpi.dealer_rollup_scd_current AS dru
            ON dru.dealer_number = cd.dealer_number
        LEFT JOIN sandbox.rds_blackbook_rollup bbvt
            ON bbvt.account_number = cd.account_number
        LEFT JOIN sandbox.kmx_approvals kmx
            ON kmx.loan_id = cd.loan_id
        WHERE cd.data_source_name != 'SPARTAN'
          AND cd.book_date IS NOT NULL
          AND cd.book_date >= {min_book_date}
    """),
    ('3e', '+ pcat (pb_customer_id, data_source_id=17)', """
        SELECT COUNT(*) AS after_pcat_join
        FROM edwnpi.los_deal_current_fact AS cd
        LEFT JOIN (
            SELECT loan_id,
                   MAX(federal_student_loan_flag) AS student_loan_flag,
                   MAX(chime_flag)                AS sandbox_chime
            FROM sandbox.student_loan_chime_flags
            GROUP BY loan_id
        ) stu ON stu.loan_id = cd.loan_id
        LEFT JOIN edwnpi.dealer_rollup_scd_current AS dru
            ON dru.dealer_number = cd.dealer_number
        LEFT JOIN sandbox.rds_blackbook_rollup bbvt
            ON bbvt.account_number = cd.account_number
        LEFT JOIN sandbox.kmx_approvals kmx
            ON kmx.loan_id = cd.loan_id
        LEFT JOIN sandbox.temp_prov_customer_credit_attributes_ragu AS pcat
            ON pcat.customerid = cd.pb_customer_id
           AND cd.data_source_id = 17
        WHERE cd.data_source_name != 'SPARTAN'
          AND cd.book_date IS NOT NULL
          AND cd.book_date >= {min_book_date}
    """),
    ('3f', '+ ccat primary borrower (pb_customer_id, data_source_id=100)', """
        SELECT COUNT(*) AS after_ccat_pb_join
        FROM edwnpi.los_deal_current_fact AS cd
        LEFT JOIN (
            SELECT loan_id,
                   MAX(federal_student_loan_flag) AS student_loan_flag,
                   MAX(chime_flag)                AS sandbox_chime
            FROM sandbox.student_loan_chime_flags
            GROUP BY loan_id
        ) stu ON stu.loan_id = cd.loan_id
        LEFT JOIN edwnpi.dealer_rollup_scd_current AS dru
            ON dru.dealer_number = cd.dealer_number
        LEFT JOIN sandbox.rds_blackbook_rollup bbvt
            ON bbvt.account_number = cd.account_number
        LEFT JOIN sandbox.kmx_approvals kmx
            ON kmx.loan_id = cd.loan_id
        LEFT JOIN sandbox.temp_prov_customer_credit_attributes_ragu AS pcat
            ON pcat.customerid = cd.pb_customer_id
           AND cd.data_source_id = 17
        LEFT JOIN sandbox.temp_los_customer_credit_attributes_ragu AS ccat
            ON ccat.customer_id = cd.pb_customer_id
           AND cd.data_source_id = 100
        WHERE cd.data_source_name != 'SPARTAN'
          AND cd.book_date IS NOT NULL
          AND cd.book_date >= {min_book_date}
    """),
    ('3g', '+ cbcat co-borrower (cb_customer_id) [same table, second alias]', """
        SELECT COUNT(*) AS after_cbcat_join
        FROM edwnpi.los_deal_current_fact AS cd
        LEFT JOIN (
            SELECT loan_id,
                   MAX(federal_student_loan_flag) AS student_loan_flag,
                   MAX(chime_flag)                AS sandbox_chime
            FROM sandbox.student_loan_chime_flags
            GROUP BY loan_id
        ) stu ON stu.loan_id = cd.loan_id
        LEFT JOIN edwnpi.dealer_rollup_scd_current AS dru
            ON dru.dealer_number = cd.dealer_number
        LEFT JOIN sandbox.rds_blackbook_rollup bbvt
            ON bbvt.account_number = cd.account_number
        LEFT JOIN sandbox.kmx_approvals kmx
            ON kmx.loan_id = cd.loan_id
        LEFT JOIN sandbox.temp_prov_customer_credit_attributes_ragu AS pcat
            ON pcat.customerid = cd.pb_customer_id
           AND cd.data_source_id = 17
        LEFT JOIN sandbox.temp_los_customer_credit_attributes_ragu AS ccat
            ON ccat.customer_id = cd.pb_customer_id
           AND cd.data_source_id = 100
        LEFT JOIN sandbox.temp_los_customer_credit_attributes_ragu AS cbcat
            ON cbcat.customer_id = cd.cb_customer_id
        WHERE cd.data_source_name != 'SPARTAN'
          AND cd.book_date IS NOT NULL
          AND cd.book_date >= {min_book_date}
    """),
    ('3h', '+ fraud table (loan_id) — full join set', """
        SELECT COUNT(*) AS after_fraud_join
        FROM edwnpi.los_deal_current_fact AS cd
        LEFT JOIN (
            SELECT loan_id,
                   MAX(federal_student_loan_flag) AS student_loan_flag,
                   MAX(chime_flag)                AS sandbox_chime
            FROM sandbox.student_loan_chime_flags
            GROUP BY loan_id
        ) stu ON stu.loan_id = cd.loan_id
        LEFT JOIN edwnpi.dealer_rollup_scd_current AS dru
            ON dru.dealer_number = cd.dealer_number
        LEFT JOIN sandbox.rds_blackbook_rollup bbvt
            ON bbvt.account_number = cd.account_number
        LEFT JOIN sandbox.kmx_approvals kmx
            ON kmx.loan_id = cd.loan_id
        LEFT JOIN sandbox.temp_prov_customer_credit_attributes_ragu AS pcat
            ON pcat.customerid = cd.pb_customer_id
           AND cd.data_source_id = 17
        LEFT JOIN sandbox.temp_los_customer_credit_attributes_ragu AS ccat
            ON ccat.customer_id = cd.pb_customer_id
           AND cd.data_source_id = 100
        LEFT JOIN sandbox.temp_los_customer_credit_attributes_ragu AS cbcat
            ON cbcat.customer_id = cd.cb_customer_id
        LEFT JOIN sandbox.temp_fraud_ragu AS fraud
            ON fraud.loan_id = cd.loan_id
        WHERE cd.data_source_name != 'SPARTAN'
          AND cd.book_date IS NOT NULL
          AND cd.book_date >= {min_book_date}
    """),
]

prev_count = None
for step_id, desc, sql in cardinality_steps:
    ok = run_step(step_id, desc, sql, conn)
    if not ok and STOP_ON_ERROR:
        break

In [ ]:
# =============================================================================
# STEP 4: UNIQUENESS CHECKS ON JOIN KEYS
# Any non-zero result means that table fans out rows in the main query.
# =============================================================================

uniqueness_steps = [
    ('4a', 'rds_blackbook_rollup — unique on account_number?',
     "SELECT COUNT(*) - COUNT(DISTINCT account_number) AS duplicate_account_numbers FROM sandbox.rds_blackbook_rollup"),
    ('4b', 'kmx_approvals — unique on loan_id?',
     "SELECT COUNT(*) - COUNT(DISTINCT loan_id) AS duplicate_loan_ids FROM sandbox.kmx_approvals"),
    ('4c', 'temp_prov_customer_credit_attributes_ragu — unique on customerid?',
     "SELECT COUNT(*) - COUNT(DISTINCT customerid) AS duplicate_customer_ids FROM sandbox.temp_prov_customer_credit_attributes_ragu"),
    ('4d', 'temp_los_customer_credit_attributes_ragu — unique on customer_id?',
     "SELECT COUNT(*) - COUNT(DISTINCT customer_id) AS duplicate_customer_ids FROM sandbox.temp_los_customer_credit_attributes_ragu"),
    ('4e', 'dealer_rollup_scd_current — unique on dealer_number?',
     "SELECT COUNT(*) - COUNT(DISTINCT dealer_number) AS duplicate_dealer_numbers FROM edwnpi.dealer_rollup_scd_current"),
    ('4f', 'student_loan_chime_flags — unique on loan_id (pre-GROUP BY)?',
     "SELECT COUNT(*) - COUNT(DISTINCT loan_id) AS duplicate_loan_ids FROM sandbox.student_loan_chime_flags"),
    ('4g', 'loan_random_numbers — unique on loan_id?',
     "SELECT COUNT(*) - COUNT(DISTINCT loan_id) AS duplicate_loan_ids FROM sandbox.loan_random_numbers"),
]

for step_id, desc, sql in uniqueness_steps:
    ok = run_step(step_id, desc, sql, conn)
    if not ok and STOP_ON_ERROR:
        break

In [ ]:
# =============================================================================
# STEP 5: WHERE CLAUSE FILTER IMPACT
# =============================================================================

# 5a: How many rows does student_loan_flag = 0 remove?
#     Rows with no CTE match (NULL) are also excluded — implicit INNER JOIN effect.
run_step('5a', 'student_loan_flag = 0 filter impact', """
    WITH student_loan AS (
        SELECT
            loan_id,
            MAX(federal_student_loan_flag) AS student_loan_flag
        FROM sandbox.student_loan_chime_flags
        GROUP BY loan_id
    )
    SELECT
        SUM(CASE WHEN stu.student_loan_flag IS NULL THEN 1 ELSE 0 END) AS no_match_in_cte,
        SUM(CASE WHEN stu.student_loan_flag = 1     THEN 1 ELSE 0 END) AS student_loan_flag_1,
        SUM(CASE WHEN stu.student_loan_flag = 0     THEN 1 ELSE 0 END) AS student_loan_flag_0,
        COUNT(*)                                                        AS total_rows
    FROM edwnpi.los_deal_current_fact AS cd
    LEFT JOIN student_loan stu ON stu.loan_id = cd.loan_id
    WHERE cd.data_source_name != 'SPARTAN'
      AND cd.book_date IS NOT NULL
      AND cd.book_date >= {min_book_date}
""", conn)

# 5b: Does application_received_date column exist on the base table?
#     The main query uses both application_received_dtm and application_received_date.
run_step('5b', 'application_received_date column existence and range', """
    SELECT
        COUNT(*)                            AS total_rows,
        COUNT(cd.application_received_date) AS non_null_app_date,
        MIN(cd.application_received_date)   AS min_app_date,
        MAX(cd.application_received_date)   AS max_app_date
    FROM edwnpi.los_deal_current_fact AS cd
    WHERE cd.data_source_name != 'SPARTAN'
      AND cd.book_date IS NOT NULL
      AND cd.book_date >= {min_book_date}
""", conn)

In [ ]:
# =============================================================================
# STEP 6: EXPRESSION VALIDATION
# Catch divide-by-zero, bad casts, and invalid values before the full query runs.
# =============================================================================

# 6a: Divide-by-zero risk in pti
#     The CASE guards against < 1, but check how many rows would hit the guard.
run_step('6a', 'pti divide-by-zero guard — zero/negative total_income rows', """
    SELECT COUNT(*) AS rows_where_total_income_zero_or_negative
    FROM edwnpi.los_deal_current_fact AS cd
    WHERE (cd.pb_monthly_income_gross_total + COALESCE(cd.cb_monthly_income_gross_total, 0)) <= 0
      AND cd.data_source_name != 'SPARTAN'
      AND cd.book_date IS NOT NULL
      AND cd.book_date >= {min_book_date}
""", conn)

# 6b: NULL or zero fraud_adjustment values
run_step('6b', 'fraud_adjustment — NULLs and invalid values', """
    SELECT
        COUNT(*)                                                   AS total_fraud_rows,
        SUM(CASE WHEN fraud_adjustment IS NULL THEN 1 ELSE 0 END)  AS null_fraud_adj,
        SUM(CASE WHEN fraud_adjustment <= 0    THEN 1 ELSE 0 END)  AS zero_or_neg_fraud_adj,
        MIN(fraud_adjustment)                                       AS min_fraud_adj,
        MAX(fraud_adjustment)                                       AS max_fraud_adj
    FROM sandbox.temp_fraud_ragu
""", conn)

# 6c: Non-numeric vantage values that would break the ::int cast
run_step('6c', 'vantage ::int cast — non-numeric values that would fail', """
    SELECT COUNT(*) AS non_numeric_vantage_rows
    FROM sandbox.temp_los_customer_credit_attributes_ragu
    WHERE vantage IS NOT NULL
      AND vantage !~ '^-?[0-9]+(\\.[0-9]+)?$'
""", conn)

In [ ]:
# =============================================================================
# STEP 7: FULL QUERY SMOKE TEST — LIMIT 100
# Runs the complete original query structure end-to-end.
# LIMIT 100 forces compilation and expression evaluation without pulling the
# full result set into memory. Error message will point to the exact problem.
# =============================================================================

run_step('7', 'Full query smoke test — LIMIT 100', """
    WITH student_loan AS (
        SELECT
            loan_id,
            MAX(federal_student_loan_flag) AS student_loan_flag,
            MAX(chime_flag)                AS sandbox_chime
        FROM sandbox.student_loan_chime_flags
        GROUP BY 1
    ),
    employment_agg AS (
        SELECT
            account_number,
            MAX(CASE WHEN employment_type = 'seasonal' THEN 1 ELSE 0 END) AS seasonal_flag,
            MAX(CASE WHEN employment_type = 'waiter'   THEN 1 ELSE 0 END) AS waiter_flag
        FROM sandbox.temp_employment_type_ragu
        GROUP BY account_number
    )
    SELECT *
    FROM edwnpi.los_deal_current_fact                                    AS cd
    LEFT JOIN student_loan stu                ON stu.loan_id             = cd.loan_id
    LEFT JOIN edwnpi.dealer_rollup_scd_current AS dru                    ON dru.dealer_number         = cd.dealer_number
    LEFT JOIN edwnpi.date_dim AS dd                                       ON cd.book_date              = dd.calendar_date
    LEFT JOIN edwnpi.date_dim AS dd_app                                   ON date(cd.application_received_dtm) = dd_app.calendar_date
    LEFT JOIN sandbox.rds_blackbook_rollup bbvt                           ON bbvt.account_number       = cd.account_number
    LEFT JOIN sandbox.kmx_approvals kmx                                   ON kmx.loan_id               = cd.loan_id
    LEFT JOIN edwnpi.dealer_attributes_pivot AS dap                       ON cd.dealer_number          = dap.dealerid AND dap.datasourceid = 17
    LEFT JOIN edwnpi.crm_dealer_dim AS cdd                                ON cdd.dealer_number         = cd.dealer_number AND cdd.current_version_flag = 1
    LEFT JOIN employment_agg AS ett                                       ON ett.account_number        = cd.account_number
    LEFT JOIN sandbox.kmx_los_new_sp lkat                                 ON lkat.loan_id              = cd.loan_id
    LEFT JOIN sandbox.temp_prov_customer_credit_attributes_ragu AS pcat   ON pcat.customerid           = cd.pb_customer_id AND cd.data_source_id = 17
    LEFT JOIN sandbox.temp_los_customer_credit_attributes_ragu  AS ccat   ON ccat.customer_id          = cd.pb_customer_id AND cd.data_source_id = 100
    LEFT JOIN sandbox.temp_los_customer_credit_attributes_ragu  AS cbcat  ON cbcat.customer_id         = cd.cb_customer_id
    LEFT JOIN sandbox.loan_random_numbers lrn                             ON lrn.loan_id               = cd.loan_id
    LEFT JOIN sandbox.temp_fraud_ragu AS fraud                            ON fraud.loan_id             = cd.loan_id
    WHERE cd.data_source_name != 'SPARTAN'
      AND cd.book_date IS NOT NULL
      AND cd.book_date >= {min_book_date}
      AND stu.student_loan_flag = 0
    LIMIT 100
""", conn)

In [ ]:
# =============================================================================
# CELL 9: CLOSE CONNECTION
# Run this cell when finished to release the Redshift connection.
# =============================================================================

conn.close()
print("Connection closed.")